# 04. Scoring and Evaluation

In this notebook, you will learn how to measure and record the quality of LLM outputs.

## What you will learn
- Adding manual scores to traces (`create_score()`)
- Numeric / Categorical / Boolean score types
- **LLM-as-a-Judge**: Automatically evaluating responses using Claude as a judge
- Building an evaluation pipeline

## What is a Score?

In Langfuse, a score is an evaluation value attached to a specific trace or span.

| Type | Example | Use case |
|------|---------|----------|
| `NUMERIC` | 0.0 ~ 1.0 | Quality score, similarity |
| `CATEGORICAL` | "good" / "bad" | Classification result |
| `BOOLEAN` | True / False | Correctness |

## What is LLM-as-a-Judge?

Instead of having humans evaluate each response manually, this technique uses a high-performing LLM (the judge model) to automatically evaluate the outputs of other LLMs.

```
[User question] + [LLM response] → [Judge LLM] → [Quality score + Reasoning]
```

## Step 1: Initial setup

In [ ]:
import os
import json
from dotenv import load_dotenv

load_dotenv()
os.environ["ANTHROPIC_API_KEY"] = os.environ["CLAUDE_API_KEY"]

from langfuse import get_client, observe
from opentelemetry.instrumentation.anthropic import AnthropicInstrumentor
from anthropic import Anthropic

AnthropicInstrumentor().instrument()
langfuse = get_client()
client = Anthropic()

print("✅ Setup complete")

## Step 2: Capture trace IDs

To attach scores, you need the ID of the target trace.
You can retrieve the current trace ID inside an `@observe()` decorated function.

In [ ]:
from opentelemetry import trace as otel_trace

captured_trace_ids = []


@observe()
def answer_question(question: str) -> dict:
    response = client.messages.create(
        model="claude-sonnet-4-6",
        max_tokens=512,
        messages=[{"role": "user", "content": question}]
    )
    
    answer = response.content[0].text
    
    # Extract trace ID from current OTel span
    span = otel_trace.get_current_span()
    ctx = span.get_span_context()
    trace_id_hex = format(ctx.trace_id, '032x')
    
    captured_trace_ids.append(trace_id_hex)
    
    return {"answer": answer, "trace_id": trace_id_hex}


qa_pairs = [
    "What is the capital of South Korea?",
    "What is the difference between a list and a tuple in Python?",
    "Explain the difference between HTTP and HTTPS.",
]

results = []
for question in qa_pairs:
    result = answer_question(question)
    results.append({"question": question, **result})
    print(f"Q: {question}")
    print(f"A: {result['answer'][:100]}...")
    print(f"Trace ID: {result['trace_id']}")
    print()

langfuse.flush()

## Step 3: Add manual scores (Numeric)

In [ ]:
# Manually assign quality scores to the first response
if results:
    langfuse.create_score(
        trace_id=results[0]["trace_id"],
        name="helpfulness",
        value=0.95,
        data_type="NUMERIC",
        comment="Clear and accurate answer"
    )
    
    langfuse.create_score(
        trace_id=results[0]["trace_id"],
        name="conciseness",
        value=0.8,
        data_type="NUMERIC",
        comment="Could be a bit more concise"
    )
    
    print(f"✅ Scores added to trace {results[0]['trace_id'][:16]}...")

langfuse.flush()

## Step 4: Add manual scores (Categorical & Boolean)

In [ ]:
if len(results) >= 2:
    # Categorical score
    langfuse.create_score(
        trace_id=results[1]["trace_id"],
        name="quality_tier",
        value="excellent",
        data_type="CATEGORICAL",
        comment="Clearly explains key differences"
    )
    
    # Boolean score
    langfuse.create_score(
        trace_id=results[1]["trace_id"],
        name="factually_correct",
        value=1,  # 1 = True, 0 = False
        data_type="BOOLEAN",
        comment="Factually correct answer"
    )
    
    print(f"✅ Scores added to trace {results[1]['trace_id'][:16]}...")

langfuse.flush()

## Step 5: Implement LLM-as-a-Judge

Use Claude as a judge to automatically evaluate the quality of LLM responses.
The judge model returns a score and reasoning in JSON format based on evaluation criteria (rubric).

In [ ]:
JUDGE_SYSTEM_PROMPT = """You are an expert evaluator who assesses the quality of AI responses.

Evaluate the response based on the following criteria:
1. **Accuracy**: Is the response factually correct? (0-10)
2. **Completeness**: Does it cover all aspects of the question? (0-10)
3. **Clarity**: Is it easy to understand and well-structured? (0-10)

You must respond only in the following JSON format:
{
  "accuracy": <0-10>,
  "completeness": <0-10>,
  "clarity": <0-10>,
  "overall": <0-10>,
  "reasoning": "<Provide 2-3 sentences explaining your evaluation>"
}"""


def judge_response(question: str, response: str) -> dict:
    """Evaluate response quality using Claude and return scores"""
    judge_prompt = f"""Please evaluate the following AI response to the given question.

Question: {question}

AI Response:
{response}"""
    
    judge_result = client.messages.create(
        model="claude-sonnet-4-6",
        max_tokens=512,
        system=JUDGE_SYSTEM_PROMPT,
        messages=[{"role": "user", "content": judge_prompt}]
    )
    
    raw_text = judge_result.content[0].text
    
    # JSON parsing
    start = raw_text.find("{")
    end = raw_text.rfind("}") + 1
    json_str = raw_text[start:end]
    return json.loads(json_str)


# Test
test_q = "What is the difference between machine learning and deep learning?"
test_a = "Machine learning is an AI technique that learns patterns from data, and deep learning is a subfield of machine learning that uses neural networks."

scores = judge_response(test_q, test_a)
print("Evaluation result:")
print(json.dumps(scores, ensure_ascii=False, indent=2))

langfuse.flush()

## Step 6: Automatically link LLM-as-a-Judge to traces

In [ ]:
@observe()
def answer_and_evaluate(question: str) -> dict:
    """Pipeline that answers questions and automatically evaluates quality"""
    
    # 1. Generate response
    response = client.messages.create(
        model="claude-sonnet-4-6",
        max_tokens=512,
        messages=[{"role": "user", "content": question}]
    )
    answer = response.content[0].text
    
    # Extract trace ID
    span = otel_trace.get_current_span()
    ctx = span.get_span_context()
    trace_id = format(ctx.trace_id, '032x')
    
    # 2. LLM-as-a-Judge evaluation
    evaluation = judge_response(question, answer)
    
    # 3. Record evaluation results as Langfuse scores
    score_names = ["accuracy", "completeness", "clarity", "overall"]
    for score_name in score_names:
        if score_name in evaluation:
            langfuse.create_score(
                trace_id=trace_id,
                name=f"judge_{score_name}",
                value=evaluation[score_name] / 10.0,  # 0-10 -> 0-1 normalization
                data_type="NUMERIC",
                comment=evaluation.get("reasoning", "")
            )
    
    return {
        "question": question,
        "answer": answer,
        "scores": evaluation,
        "trace_id": trace_id
    }


evaluation_questions = [
    "What is Python's GIL (Global Interpreter Lock)?",
    "What are the main differences between REST API and GraphQL?",
]

for question in evaluation_questions:
    result = answer_and_evaluate(question)
    print(f"Question: {result['question']}")
    print(f"Overall score: {result['scores'].get('overall', 'N/A')}/10")
    print(f"Reasoning: {result['scores'].get('reasoning', '')}")
    print(f"Trace ID: {result['trace_id'][:16]}...")
    print()

langfuse.flush()

> **Check in the dashboard:**
> Click on a trace and you will find the **Scores** tab at the bottom.
> You can see the recorded scores: `judge_accuracy`, `judge_completeness`, `judge_clarity`, and `judge_overall`.
>
> In the **Scores** menu, you can analyze score distributions and trends over time.

## Summary

| Concept | Description |
|---------|-------------|
| `create_score()` | Add evaluation scores to a trace |
| `NUMERIC` | Continuous numeric score (0~1, 0~10, etc.) |
| `CATEGORICAL` | Categorical label ("excellent", "poor", etc.) |
| `BOOLEAN` | Binary judgment (1=True, 0=False) |
| LLM-as-a-Judge | Automated evaluation using an LLM as a judge |

Next: **05_datasets_and_experiments.ipynb** -- Systematic experiment management